In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    accuracy_score, f1_score)
from sklearn.metrics import ConfusionMatrixDisplay


## Google-bert-base-chinese

In [ ]:
filepath = './asbc_eval_results/google-bert-bert-base-chinese-DottedWSD/wsd/google-bert-bert-base-chinese-DottedWSD_wsd_eval.pkl'

with open(filepath, 'rb') as f:
    google = pickle.load(f)

len(google)

In [ ]:
google.keys()

In [ ]:
google['metadata']

In [ ]:
google_df = google['by_instance']['df']
google_df.head()

## Meta-llama-3.2-3B

In [ ]:
filepath = './asbc_eval_results/meta-llama-Llama-3.2-3B-DottedWSD/wsd/meta-llama-Llama-3.2-3B-DottedWSD_wsd_eval.pkl'

with open(filepath, 'rb') as f:
    llama = pickle.load(f)
    

In [ ]:
llama_df = llama['by_instance']['df']
llama_df.head()

## Separate Acc. and Kappa

In [ ]:
if False in google_df.label == llama_df.label:
  print('Labels are not the same')

In [ ]:
# compare accuracy respectively
google_accuracy = accuracy_score(google_df.label, google_df.prediction)
llama_accuracy = accuracy_score(llama_df.label, llama_df.prediction)
print(f'Google accuracy (label, prediction): {google_accuracy}')
print(f'Llama accuracy (label, prediction): {llama_accuracy}')

In [ ]:
# annotation agreement
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(google_df.prediction, llama_df.prediction)
print(f'Kappa: {kappa}')

In [ ]:
# count the percentage of the same predictions
same_prediction = google_df.prediction == llama_df.prediction
same_prediction = same_prediction.sum() / len(same_prediction)
print(f'Percentage of same predictions: {same_prediction}')

## Disagreed instances

In [ ]:
## extract rows where the predictions are different
df = google_df.copy()
df['llama_prediction'] = llama_df.prediction
df.rename(columns={'prediction': 'google_prediction'}, inplace=True)


In [ ]:
df.head()

In [ ]:
# filter out rows where the predictions are different
diff = df[df.google_prediction != df.llama_prediction]
len(diff)

In [ ]:
# save the different predictions
diff.to_csv('asbc_diff_predictions.csv', index=False, encoding='utf-8')